# masrafAI — 25k açıklama toplu üretimi (Kaggle + vLLM + Qwen3-8B)

Dataset: **kaggle_v8_yukleme** (25 batch + durum.json + 3 py).
Gözetimsiz koşu için: **Save Version → Save & Run All (commit)**.
Süre beklentisi: ~5,1 saat (147 sn / 200 kayıt ölçümünden).

## Hücre 1 — kurulum

In [ ]:
!pip install -q vllm hf_transfer
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Hücre 2 — ortam (HF_TOKEN)

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN ayarlandi")
except Exception as e:
    print("HF_TOKEN yok, anonim indirme:", e)

## Hücre 3 — dosyaları kopyala + doğrula

Beklenen: `batch: 25 / 25`, `py: 3`, `ciktilar: 0 []`, `core: 179584 byte | yeni kurallar: True`.
`aday sayisi: 2` -> eski kod dataset'i hala ekli, Input panelinden cikar.

In [ ]:
import shutil, glob, os, json

adaylar = [os.path.dirname(p) for p in
           glob.glob("/kaggle/input/**/aciklama_uretim_core.py", recursive=True)]
assert adaylar, "v8 dataset'i bulunamadi - notebook'a ekli mi? !find /kaggle/input -maxdepth 3"
KAYNAK = adaylar[0]
HEDEF  = "/kaggle/working/aciklama_25k"
os.makedirs(HEDEF, exist_ok=True)
print("kaynak:", KAYNAK, "| aday sayisi:", len(adaylar))
assert len(adaylar) == 1, f"BIRDEN COK kod kaynagi bagli: {adaylar} - eski dataset'leri cikar"

for f in glob.glob(f"{KAYNAK}/**/batch_[0-9][0-9][0-9][0-9].json", recursive=True) + \
         glob.glob(f"{KAYNAK}/**/durum.json", recursive=True):
    shutil.copy(f, HEDEF)
py = sorted(glob.glob(f"{KAYNAK}/*.py"))
for f in py:
    shutil.copy(f, "/kaggle/working/")

b = sorted(glob.glob(f"{HEDEF}/batch_[0-9][0-9][0-9][0-9].json"))
c = glob.glob(f"{HEDEF}/*_ciktilar.json")
core = open("/kaggle/working/aciklama_uretim_core.py", encoding="utf-8").read()
kurallar_tamam = all(k in core for k in ("vurgu_fazlasi", "tema_halusinasyonu", "_KATEGORI_GENEL_AD"))
print("batch   :", len(b), "/ 25")
print("durum   :", os.path.exists(f"{HEDEF}/durum.json"))
print("py      :", [os.path.basename(p) for p in py])
print("ciktilar:", len(c), c)
print("tarih   :", json.load(open(b[0]))[0].get("fatura_tarihi"))
print("core    :", len(core), "byte |", "yeni kurallar:", kurallar_tamam)
assert len(b) == 25, f"batch sayisi {len(b)}, 25 olmali"
assert kurallar_tamam, "ESKI core kopyalanmis - eski kod dataset'ini cikar"

## Hücre 4 — vLLM sunucusu

Model indirmesi 10-20 dk sürer ve bu sırada hücre **hiçbir şey basmaz** (log `vllm.log`'a gider) —
donmuş değildir. Hata alırsan: `!tail -50 /kaggle/working/vllm.log`

In [ ]:
import subprocess, time, requests

MODEL = "Qwen/Qwen3-8B"          # yedek: "Qwen/Qwen3-4B-Instruct-2507"
log = open("/kaggle/working/vllm.log", "w")
cmd = ["python", "-m", "vllm.entrypoints.openai.api_server",
       "--model", MODEL,
       "--dtype", "half",
       "--tensor-parallel-size", "2",
       "--max-model-len", "2048",
       "--gpu-memory-utilization", "0.92",
       "--enable-prefix-caching",
       "--generation-config", "vllm",
       "--enforce-eager",
       "--port", "8000"]
sunucu = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

url = "http://localhost:8000/v1/models"
hazir = False
for i in range(360):                     # 60 dk
    if sunucu.poll() is not None:
        print("SUNUCU OLDU, exit =", sunucu.returncode)
        break
    try:
        if MODEL in str(requests.get(url, timeout=5).json()):
            print("HAZIR, gecen sure (sn) =", i * 10)
            hazir = True
            break
    except Exception:
        pass
    time.sleep(10)
else:
    print("60 dk doldu - !tail -50 /kaggle/working/vllm.log")

# /v1/models 200 donse de uretim calismiyor olabilir: gercek istek at.
if hazir:
    r = requests.post("http://localhost:8000/v1/chat/completions",
                      json={"model": MODEL, "max_tokens": 16,
                            "messages": [{"role": "user", "content": "Merhaba de."}]},
                      timeout=180)
    print("duman testi:", r.status_code, r.json()["choices"][0]["message"]["content"][:80])

## Hücre 5 — tam üretim (25.000 kayıt, ~5 saat)

Log dosyaya yönlendiriliyor: runner fatura başına bir satır basıyor (25.000 satır),
notebook çıktısına gömülmesin. Kesilirse **aynı hücreyi tekrar çalıştır** —
`*_ciktilar.json` dosyalarından kaldığı yerden devam eder.

In [ ]:
%%time
!python -u /kaggle/working/aciklama_toplu_uret.py \
    --cikti-dizini /kaggle/working/aciklama_25k \
    --saglayici vllm --host http://localhost:8000/v1 \
    --model Qwen/Qwen3-8B \
    --sicaklik-tavani 0.9 --cooldown-min 0 --workers 16 \
    --batch 1-25 > /kaggle/working/uretim.log 2>&1
!tail -60 /kaggle/working/uretim.log

## Hücre 6 — özet + indirilecek paket

try/except ile sarılı: üretim yarım kalsa bile bu hücre patlamaz, commit yeşil biter
ve `/kaggle/working` versiyon çıktısı olarak kaydedilir.

In [ ]:
import json, glob, collections, zipfile, os
try:
    dosyalar = sorted(glob.glob("/kaggle/working/aciklama_25k/*_ciktilar.json"))
    toplam, ih, kat = 0, collections.Counter(), collections.Counter()
    for f in dosyalar:
        C = json.load(open(f))
        toplam += len(C)
        for v in C.values():
            kat[v["aciklama_kategorisi"]] += 1
            for x in (v.get("kalan_ihlaller") or []):
                ih[x] += 1
    print("cikti dosyasi:", len(dosyalar), "/ 25   uretilen kayit:", toplam, "/ 25000")
    print("kategori:", dict(kat))
    print("kalan ihlaller:", dict(ih.most_common(12)))

    with zipfile.ZipFile("/kaggle/working/ciktilar_25k.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for f in dosyalar:
            z.write(f, os.path.basename(f))
        if os.path.exists("/kaggle/working/uretim.log"):
            z.write("/kaggle/working/uretim.log", "uretim.log")
    print("zip:", os.path.getsize("/kaggle/working/ciktilar_25k.zip") // 1024, "KB")
except Exception as e:
    print("ozet basarisiz (uretim yarim kalmis olabilir):", type(e).__name__, e)

!python /kaggle/working/aciklama_analiz.py --cikti-dizini /kaggle/working/aciklama_25k || true